# **CALIFORNIA HOUSING VALUE PREDICTION**

In [19]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim


# **LOADING DATA**

In [20]:
df=pd.read_csv("/content/sample_data/california_housing_train.csv")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0


# **SETTING INPUT AND TARGET**

In [21]:
y=df.median_house_value
x=df.drop(columns=["median_house_value"])

# **PREPROCESSING**

In [22]:
y=torch.tensor(y.values,dtype=torch.float32)
x=torch.tensor(x.values,dtype=torch.float32)
x.dtype,y.dtype

(torch.float32, torch.float32)

In [23]:
y = y.view(-1, 1)

In [24]:
y

tensor([[ 66900.],
        [ 80100.],
        [ 85700.],
        ...,
        [103600.],
        [ 85800.],
        [ 94600.]])

In [25]:
x

tensor([[-114.3100,   34.1900,   15.0000,  ..., 1015.0000,  472.0000,
            1.4936],
        [-114.4700,   34.4000,   19.0000,  ..., 1129.0000,  463.0000,
            1.8200],
        [-114.5600,   33.6900,   17.0000,  ...,  333.0000,  117.0000,
            1.6509],
        ...,
        [-124.3000,   41.8400,   17.0000,  ..., 1244.0000,  456.0000,
            3.0313],
        [-124.3000,   41.8000,   19.0000,  ..., 1298.0000,  478.0000,
            1.9797],
        [-124.3500,   40.5400,   52.0000,  ...,  806.0000,  270.0000,
            3.0147]])

In [26]:
x_mean = x.mean(dim=0)
x_std = x.std(dim=0)
x = (x - x_mean) / x_std

In [27]:
x

tensor([[ 2.6193, -0.6715, -1.0796,  ..., -0.3612, -0.0760, -1.2525],
        [ 2.5395, -0.5732, -0.7618,  ..., -0.2619, -0.0994, -1.0815],
        [ 2.4946, -0.9054, -0.9207,  ..., -0.9553, -0.9992, -1.1701],
        ...,
        [-2.3628,  2.9077, -0.9207,  ..., -0.1617, -0.1176, -0.4466],
        [-2.3628,  2.8890, -0.7618,  ..., -0.1146, -0.0604, -0.9978],
        [-2.3878,  2.2995,  1.8599,  ..., -0.5433, -0.6013, -0.4553]])

In [28]:
y_mean = y.mean()
y_std = y.std()
y = (y - y_mean) / y_std

In [29]:
y

tensor([[-1.2105],
        [-1.0967],
        [-1.0484],
        ...,
        [-0.8941],
        [-1.0476],
        [-0.9717]])

In [30]:
print(x.shape, y.shape)

torch.Size([17000, 8]) torch.Size([17000, 1])


# **MODEL CREATION**

In [31]:
class FakeModel(nn.Module):
  def __init__(self,input_dim,hidden_dim,output_dim):
    super().__init__()
    self.linear1=nn.Linear(in_features=input_dim,out_features=hidden_dim)
    self.relu=nn.ReLU()
    self.linear2=nn.Linear(in_features=hidden_dim,out_features=output_dim)
  def forward(self,x):
    x=self.linear1(x)
    x=self.relu(x)
    x=self.linear2(x)
    return x

In [32]:
model=FakeModel(input_dim=8,hidden_dim=16,output_dim=1)
optimizer=torch.optim.Adam(model.parameters(),lr=0.01)
loss_fn=nn.MSELoss()

# **TRAINING LOOP**

In [33]:
epochs=100
for epoch in range(epochs):
  y_hat=model(x)
  loss=loss_fn(y_hat,y)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()
  if epoch%10==0:
    print(f"epoch {epoch}: {loss.item():.2f}")

epoch 0: 1.17
epoch 10: 0.74
epoch 20: 0.51
epoch 30: 0.41
epoch 40: 0.37
epoch 50: 0.34
epoch 60: 0.32
epoch 70: 0.31
epoch 80: 0.30
epoch 90: 0.29


# **MODEL EVALUATION**

In [34]:
df2=pd.read_csv("/content/sample_data/california_housing_test.csv")
df2.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-122.05,37.37,27.0,3885.0,661.0,1537.0,606.0,6.6085,344700.0
1,-118.30,34.26,43.0,1510.0,310.0,809.0,277.0,3.5990,176500.0
2,-117.81,33.78,27.0,3589.0,507.0,1484.0,495.0,5.7934,270500.0
3,-118.36,33.82,28.0,67.0,15.0,49.0,11.0,6.1359,330000.0
4,-119.67,36.33,19.0,1241.0,244.0,850.0,237.0,2.9375,81700.0


In [35]:
x_test=df2.drop(columns=["median_house_value"])
y_test=df2["median_house_value"]

In [36]:
y_test=torch.tensor(y_test,dtype=torch.float32)
y_test

tensor([344700., 176500., 270500.,  ...,  62000., 162500., 500001.])

In [37]:
y_test=y_test.view(-1,1)

In [38]:
x_test=torch.tensor(x_test.values,dtype=torch.float32)

In [39]:
x_test = (x_test - x_mean) / x_std

In [40]:
y_test = (y_test - y_mean) / y_std

In [41]:
x_test.shape,y_test.shape

(torch.Size([3000, 8]), torch.Size([3000, 1]))

In [42]:
x_test

tensor([[-1.2407,  0.8163, -0.1263,  ...,  0.0936,  0.2725,  1.4280],
        [ 0.6294, -0.6387,  1.1449,  ..., -0.5406, -0.5831, -0.1491],
        [ 0.8738, -0.8633, -0.1263,  ...,  0.0474, -0.0162,  1.0009],
        ...,
        [-0.0688,  0.3157, -1.4769,  ..., -0.6417, -0.7314, -0.8354],
        [ 1.2179, -0.7136,  0.9065,  ..., -1.2054, -1.2671, -0.3211],
        [-0.0339, -0.5639,  1.0654,  ..., -0.5894, -0.6273,  2.4512]])

In [43]:
y_test

tensor([[ 1.1846],
        [-0.2656],
        [ 0.5449],
        ...,
        [-1.2528],
        [-0.3863],
        [ 2.5236]])

In [44]:
model.eval()

FakeModel(
  (linear1): Linear(in_features=8, out_features=16, bias=True)
  (relu): ReLU()
  (linear2): Linear(in_features=16, out_features=1, bias=True)
)

In [45]:
model.eval()

with torch.no_grad():
    y_pred = model(x_test)
    test_loss = loss_fn(y_pred, y_test)

print(test_loss.item())

0.29309335350990295
